In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Single Head Attention

In [6]:
def attention(Q, K, V):
  N = Q.shape[0]
  d_k = Q.shape[-1]

  scale = 1 / math.sqrt(d_k)
  S = Q @ K.mT * scale

  mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
  S = S.masked_fill(mask, float('-inf'))

  P = torch.softmax(S, dim = -1)
  O = P @ V
  return O

In [7]:
N = 32
d_model = 512
d_k = d_q = d_v = d_model

In [8]:
K = torch.rand(N, d_k)
Q = torch.rand(N, d_q)
V = torch.rand(N, d_v)

In [11]:
O = attention(Q, K, V)
ref = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

In [12]:
torch.allclose(O, ref, atol=1e-5)

True

# Multi-Head Attention

In [ ]:
class MultiHeadedAttention(nn.Module):
  def __init__(self, num_heads, d_model, bias=False):
    super().__init__()
    assert d_model % num_heads == 0, "model dimension must be divisible by the number of heads"

    self.num_heads = num_heads
    self.d_model = d_model
    self.d_k = d_model // num_heads

    # weight matrix projections
    self.W_Q = nn.Linear(d_model, d_model, bias=bias)
    self.W_K = nn.Linear(d_model, d_model, bias=bias)
    self.W_V = nn.Linear(d_model, d_model, bias=bias)
    self.W_O = nn.Linear(d_model, d_model, bias=bias)

  # attention
  def scaled_dot_product_attention(self, Q, K, V, mask=None):
    scale = 1 / math.sqrt(self.d_k)
    S = Q @ K.transpose(-2, -1) * scale

    if mask is not None:
      S = S.masked_fill(mask, float('-inf'))

    P = torch.softmax(S, dim = -1)
    O = P @ V
    return O

  # forward pass
  def forward(self, x, mask=None):
    # apply projections
    Q = self.W_Q(x)
    K = self.W_K(x)
    V = self.W_V(x)

    # split feature dim into heads
    Q = Q.view(*x.shape[:-1], self.num_heads, self.d_k).transpose(-3, -2)
    K = K.view(*x.shape[:-1], self.num_heads, self.d_k).transpose(-3, -2)
    V = V.view(*x.shape[:-1], self.num_heads, self.d_k).transpose(-3, -2)

    O = self.scaled_dot_product_attention(Q, K, V, mask)
    # undo transpose and reshape, merges d_k and h back to d_model, reshape left it non-contiguous
    O = O.transpose(-3, -2).reshape(*x.shape[:-1], self.d_model)
    O = self.W_O(O)

    return O

In [ ]:
h = 8
mha = MultiHeadedAttention(h, d_model)

In [ ]:
B = 4
x = torch.rand(N, d_model)
x_batched = torch.rand(B, N, d_model)

print(mha(x).shape, ) 
print(mha(x_batched).shape)

In [ ]:
print(torch.allclose(mha(x_batched[0]), mha(x_batched)[0], atol=1e-5))

In [ ]:
mask = torch.triu(torch.ones(N, N, dtype=torch.bool), diagonal=1)
mask

In [ ]:
x2 = x.clone(); x2[N//2:] = torch.rand(N - N//2, d_model)
x2

In [ ]:
torch.allclose(mha(x, mask)[:N//2], mha(x2, mask)[:N//2], atol=1e-5)